In [3]:
import pandas as pd
import torch
import os
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()

def fileexistorNot(save_path, symbol):
    if os.path.exists(save_path):
        print(f"{symbol} Exist, so no need for full sentiment analysis")
        old_df = pd.read_csv(save_path)

        if not old_df.empty:
            last_date = old_df["Date"][0]
            return last_date, old_df
        else:
            return None, old_df
    else:
        print(f"{symbol} is not present, so need full sentiment analysis")
        return None, None

def analyze_text(text):
    if not isinstance(text, str) or text.strip() == "":
        return 0.0

    inputs = tokenizer(
        text.strip(),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)[0].numpy()

    return float(probs[2] - probs[0])


query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails["Symbol"]

stock_ignored = ["CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "AMFIPO", "NMLBS", "ILFCPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]


for symbol in symbols:

    print(f"\nScoring {symbol} stock...")

    if symbol in stock_ignored:
        print(f"{symbol} is in ignore list!")
        continue

    if symbol in stock_no_news:
        print(f"{symbol} is in stock no news list!")
        continue

    save_path = f"../DATA-HTML-STOCK/STOCKSENTIMENT/{symbol}news_sentiment.csv"
    news_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"

    last_date, old_df = fileexistorNot(save_path, symbol)

    if not os.path.exists(news_path):
        print(f"{symbol} news file not found")
        continue

    news_df = pd.read_csv(news_path)

    new_rows = []

    for index, row in news_df.iterrows():

        date = row.get("Date")

        if last_date is not None and date == last_date:
            print(f"Reached old_date: {last_date}, new_date: {date}")
            break

        headline = row.get("Headline", "")
        full_content = row.get("Full Content", "")
        text = str(headline) + " " + str(full_content)

        score = analyze_text(text)

        row_copy = row.copy()
        row_copy["Sentiment_Score"] = score
        new_rows.append(row_copy)

        print(index, end=" ")

    if len(new_rows) > 0:
        new_df = pd.DataFrame(new_rows)

        if old_df is not None:
            final_df = pd.concat([new_df, old_df], ignore_index=True)
        else:
            final_df = new_df

        final_df.to_csv(save_path, index=False)
        print(f"\n{symbol} Done")
    else:
        print(f"\nNo new news for {symbol}")

print("\nAll Done.")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Scoring ADBL stock...
ADBL Exist, so no need for full sentiment analysis
Reached old date 2026-02-17, stopping.

No new news for ADBL.

Scoring API stock...
API Exist, so no need for full sentiment analysis
Reached old date 2026-03-01, stopping.

No new news for API.

Scoring HATH stock...
HATH Exist, so no need for full sentiment analysis
Reached old date 2020-08-28, stopping.

No new news for HATH.

Scoring HATHPO stock...
HATHPO is in stock no news list!

Scoring AKPL stock...
AKPL Exist, so no need for full sentiment analysis
Reached old date 2025-12-08, stopping.

No new news for AKPL.

Scoring AHPC stock...
AHPC Exist, so no need for full sentiment analysis
Reached old date 2026-01-01, stopping.

No new news for AHPC.

Scoring ALICL stock...
ALICL Exist, so no need for full sentiment analysis
Reached old date 2026-02-02, stopping.

No new news for ALICL.

Scoring ALICLP stock...
ALICLP Exist, so no need for full sentiment analysis
Reached old date 2022-09-07, stopping.

No new n